In [1]:
import os
os.chdir("../")
%pwd

'/Users/prasad/Learning/Projects/Kidney-Disease-Classification-Project'

In [14]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data_dir: Path
    params_image_size: list
    params_epochs: int
    params_batch_size: int
    params_augmentation: bool


In [15]:
from cnnClassifierKidneyDisease.constants import *
from cnnClassifierKidneyDisease.utils.common import read_yaml, create_directories

In [16]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path: Path = CONFIG_FILE_PATH,
        params_file_path: Path = PARAMS_FILE_PATH) -> None:
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        
        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        training_config = self.config.model_training
        prepare_base_model_config = self.config.prepare_base_model
        training_data = Path(self.config.data_ingestion.unzip_dir)
        
        create_directories([training_config.root_dir])

        training_config = TrainingConfig(
            root_dir=training_config.root_dir,
            trained_model_path=training_config.trained_model_path,
            updated_base_model_path=prepare_base_model_config.updated_base_model_path,
            training_data_dir=training_data,
            params_image_size=self.params.IMAGE_SIZE,
            params_epochs=self.params.EPOCHS,
            params_batch_size=self.params.BATCH_SIZE,
            params_augmentation=self.params.AUGMENTATION
        )

        return training_config

In [17]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
from cnnClassifierKidneyDisease.logging import logger
import time

In [18]:
class ModelTraining:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(self.config.updated_base_model_path)
        logger.info("Base model loaded successfully.")

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )
        
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data_dir,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                rotation_range=40,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data_dir,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)
        logger.info(f"Trained Model saved at: {path}")

    def train(self, callbacks_list: list):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_data=self.valid_generator,
            validation_steps=self.validation_steps,
            callbacks=callbacks_list
        )

        self.save_model(path=self.config.trained_model_path, model=self.model)



In [20]:
try:
    config_manager = ConfigurationManager()
    model_training_config = config_manager.get_training_config()
    model_training = ModelTraining(config=model_training_config)
    model_training.get_base_model()
    model_training.train_valid_generator()
    model_training.train(callbacks_list=None)
except Exception as e:
    logger.error(f"An error occurred during base model training: {str(e)}")

[2026-05-17 12:13:31,164: INFO: common: YAML file loaded successfully: config/config.yaml]
[2026-05-17 12:13:31,166: INFO: common: YAML file loaded successfully: params.yaml]
[2026-05-17 12:13:31,167: INFO: common: Directory created successfully: artifacts]
[2026-05-17 12:13:31,169: INFO: common: Directory created successfully: artifacts/model_training]
[2026-05-17 12:13:31,328: INFO: 2475635455: Base model loaded successfully.]
Found 2489 images belonging to 1 classes.
Found 9957 images belonging to 1 classes.
 93/622 [===>..........................] - ETA: 8:41 - loss: 101.0344 - accuracy: 0.2560

KeyboardInterrupt: 